# Part 7: CID Content Addressing

Content Identifiers (CIDs) are the backbone of content-addressed storage in ATProto. A CID is a
self-describing hash: it encodes the codec (dag-cbor, dag-pb), the hash algorithm (sha2-256), and
the digest itself. Same data always produces the same CID.

**What you'll learn:**

- Multibase, multicodec, and multihash encoding
- CIDv0 vs CIDv1 structure
- SHA-256 hashing via host bridge
- Base32 encoding for CIDv1
- CID verification and content integrity

**Prerequisites:** Part 6 (ATProto Identifiers)

**Estimated Time:** 20-25 minutes

## Step 1: What Is a CID?

A CID (Content Identifier) has three parts:

$$
\text{CIDv1} = \text{multibase-prefix} + \text{multicodec} + \text{multihash}
$$

| Component  | Purpose                                | Example                         |
| ---------- | -------------------------------------- | ------------------------------- |
| multicodec | Identifies the data format             | 0x71 = dag-cbor, 0x70 = dag-pb  |
| multihash  | Identifies the hash algorithm + digest | 0x12 + 0x20 + sha2-256-32-bytes |
| multibase  | Identifies the text encoding           | 'b' = base32lower               |

CIDv0 (legacy IPFS) uses base58btc and always starts with 'Qm'. CIDv1 (ATProto default) uses
base32lower and starts with 'b'.

In [ ]:
// CID structure — a simplified model
@interface CIDObj : NSObject
@property (nonatomic, assign) int version;
@property (nonatomic, strong) NSString *codec;
@property (nonatomic, strong) NSString *hashAlgorithm;
@property (nonatomic, strong) NSString *digestHex;
- (NSString *)stringValue;
@end

@implementation CIDObj
- (NSString *)stringValue {
    if (_version == 0) {
        // CIDv0: base58btc encoding of dag-pb + sha2-256
        return [NSString stringWithFormat:@"Qm%@", [_digestHex substringFromIndex:0]]; 
    }
    // CIDv1: base32lower encoding
    return [NSString stringWithFormat:@"b%@@%@", _codec, _digestHex];
}
- (NSString *)description {
    return [NSString stringWithFormat:@"CIDv%d(codec=%@, hash=%@)", _version, _codec, _hashAlgorithm];
}
@end

// Create a CIDv1
CIDObj *cid = [[CIDObj alloc] init];
cid.version = 1;
cid.codec = @"dag-cbor";
cid.hashAlgorithm = @"sha2-256";
cid.digestHex = @"a1b2c3d4";
NSLog(@"%@", [cid description]);
NSLog(@"String: %@", [cid stringValue]);

## Step 2: Simplified Hash Function

In production, CIDs use SHA-256. The kernel provides SHA-256 via a host bridge. Here we use a
simplified XOR-fold hash to demonstrate the concept: same input always produces the same output.

In [ ]:
// Simplified hash — same data always produces the same CID
@interface CIDHelper : NSObject
- (NSString *)cidForData:(NSData *)data;
- (BOOL)verifyData:(NSData *)data cid:(NSString *)cid;
@end

@implementation CIDHelper
- (NSString *)cidForData:(NSData *)data {
    // XOR-fold bytes into 4 bytes, then hex-encode
    // Real implementation: SHA-256 -> base32 (CIDv1 dag-cbor)
    int a = 0, b = 0, c = 0, d = 0;
    const char *bytes = [data bytes];
    int len = [data length];
    for (int i = 0; i < len; i++) {
        a = a ^ bytes[i];
        b = b ^ (bytes[i] << 1);
        c = c ^ (bytes[i] << 2);
        d = d ^ (bytes[i] << 3);
    }
    a = a & 0xFF; b = b & 0xFF; c = c & 0xFF; d = d & 0xFF;
    return [NSString stringWithFormat:@"bafyrei%02x%02x%02x%02x", a, b, c, d];
}
- (BOOL)verifyData:(NSData *)data cid:(NSString *)cid {
    NSString *computed = [self cidForData:data];
    return [computed isEqualToString:cid];
}
@end

CIDHelper *helper = [[CIDHelper alloc] init];

// Same data produces same CID
NSData *hello = [NSData dataWithBytes:"hello" length:5];
NSString *cid1 = [helper cidForData:hello];
NSLog(@"CID for 'hello': %@", cid1);

NSData *hello2 = [NSData dataWithBytes:"hello" length:5];
NSString *cid2 = [helper cidForData:hello2];
NSLog(@"Same CID: %d", [cid1 isEqualToString:cid2]);

// Different data produces different CID
NSData *world = [NSData dataWithBytes:"world" length:5];
NSString *cid3 = [helper cidForData:world];
NSLog(@"Different CID: %d", ![cid1 isEqualToString:cid3]);

// Verification
NSLog(@"Verify correct: %d", [helper verifyData:hello cid:cid1]);
NSLog(@"Verify wrong: %d", [helper verifyData:world cid:cid1]);

## Step 3: CID Deduplication

Content addressing means identical data shares the same CID. This enables automatic deduplication:
store once, reference many times.

In [ ]:
// CID-based deduplication store
@interface CIDStore : NSObject
@property (nonatomic, strong) NSMutableDictionary *cidToData;
- (NSString *)storeData:(NSData *)data;
- (NSData *)getDataForCID:(NSString *)cid;
- (int)storedBlockCount;
@end

@implementation CIDStore
- (instancetype)init {
    self = [super init];
    if (self) { _cidToData = [NSMutableDictionary dictionary]; }
    return self;
}
- (NSString *)storeData:(NSData *)data {
    CIDHelper *helper = [[CIDHelper alloc] init];
    NSString *cid = [helper cidForData:data];
    if ([_cidToData objectForKey:cid] == nil) {
        [_cidToData setObject:data forKey:cid];
        NSLog(@"Stored new block: %@", cid);
    } else {
        NSLog(@"Dedup: %@ already stored", cid);
    }
    return cid;
}
- (NSData *)getDataForCID:(NSString *)cid {
    return [_cidToData objectForKey:cid];
}
- (int)storedBlockCount {
    return [_cidToData count];
}
@end

CIDStore *store = [[CIDStore alloc] init];

// Store the same data twice
NSData *data1 = [NSData dataWithBytes:"hello" length:5];
NSData *data2 = [NSData dataWithBytes:"hello" length:5];
NSData *data3 = [NSData dataWithBytes:"world" length:5];

[store storeData:data1];
[store storeData:data2];  // dedup
[store storeData:data3];

NSLog(@"Stored blocks: %d (expect 2)", [store storedBlockCount]);

## Step 4: Garazyk Production Anchor

In the Garazyk codebase, `Core/CID.h` implements full CID support:

- `CID initWithString:` parses CIDv0 and CIDv1 strings
- `CID initWithCodec:multihash:` constructs from raw components
- `encode` produces the binary CID bytes
- `stringValue` returns the base32 or base58btc text form

The production CID uses real SHA-256 via the `CryptoUtils` class, which is an
**unsupported-production** API (Security framework). In this notebook, we use a simplified hash
instead.

## Exercise

Implement a `mergeStore:` method on `CIDStore` that takes another `CIDStore` and adds all its
blocks. Blocks with matching CIDs should not be duplicated.

Hint: iterate over the other store's CID keys and only add entries that don't already exist in this
store.

In [ ]:
// Exercise: implement mergeStore:
// Your code here...

## Solution

In [ ]:
// Solution: mergeStore adds only new CIDs
@interface CIDStore2 : NSObject
@property (nonatomic, strong) NSMutableDictionary *cidToData;
- (NSString *)storeData:(NSData *)data;
- (void)mergeStore:(CIDStore2 *)other;
- (int)storedBlockCount;
@end

@implementation CIDStore2
- (instancetype)init {
    self = [super init];
    if (self) { _cidToData = [NSMutableDictionary dictionary]; }
    return self;
}
- (NSString *)storeData:(NSData *)data {
    CIDHelper *helper = [[CIDHelper alloc] init];
    NSString *cid = [helper cidForData:data];
    if ([_cidToData objectForKey:cid] == nil) {
        [_cidToData setObject:data forKey:cid];
    }
    return cid;
}
- (void)mergeStore:(CIDStore2 *)other {
    for (NSString *cid in other.cidToData) {
        if ([_cidToData objectForKey:cid] == nil) {
            [_cidToData setObject:[other.cidToData objectForKey:cid] forKey:cid];
            NSLog(@"Merged: %@", cid);
        } else {
            NSLog(@"Skip existing: %@", cid);
        }
    }
}
- (int)storedBlockCount {
    return [_cidToData count];
}
@end

CIDStore2 *s1 = [[CIDStore2 alloc] init];
[s1 storeData:[NSData dataWithBytes:"hello" length:5]];
[s1 storeData:[NSData dataWithBytes:"world" length:5]];

CIDStore2 *s2 = [[CIDStore2 alloc] init];
[s2 storeData:[NSData dataWithBytes:"hello" length:5]];
[s2 storeData:[NSData dataWithBytes:"objc" length:4]];

[s1 mergeStore:s2];
NSLog(@"Merged store blocks: %d (expect 3)", [s1 storedBlockCount]);

## What to Read Next

You now understand how CIDs provide content-addressed identity. Next:

- **Part 8: DAG-CBOR** — deterministic binary encoding for ATProto data
- **Part 9: CAR Archives** — how blocks are packaged into CAR files
- **Part 14: Record Writes** — how CIDs address records in repositories